# 4a evaluation quicklook

Quicklook of the outputs of the **4a statistics analysis** (`climatereconstructionai/evaluate_4a.py`) — the per-grid-cell evaluation of the NN downscaling output against the reference data.

The 4a analysis runs either standalone (`crai-evaluate-4a` / `input/slurm-scripts/4a_submit_evaluation.sh`) or integrated into the model evaluation (`crai-evaluate --run-4a-stats`, see `run_4a_stats` in `climatereconstructionai/evaluate.py`). This notebook reads its output directory and shows:

- the scalar statistics file (`*_stats.txt`),
- the per-grid-cell maps (`*_maps.nc`: bias, MSE/RMSE, explained variance, correlation, variance ratio, fraction of gap closed, error quantiles, plus the deseasonalized anomaly maps if `--deseasonalize` was used),
- the figures written by the script (`*.png`: map+histogram panels, ROC/AUC, QQ plot, point/area time series),
- an interactive point/area time series (default: Hamburg), recomputed from the raw NN/reference files with the packaged `evaluate_4a` helpers.


In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- 4a output directory ---------------------------------------------------
# Default location written by the integrated run (crai-evaluate --run-4a-stats):
evaluation_dirs = "./evaluations/"
config_name = "t2m"
run_name = "cerra-germany-0.05x0.05-el5-single-l0"
stats_dir = Path(
    os.environ.get(
        "CRAI_4A_STATS_DIR",
        f"{evaluation_dirs}/{config_name}/{run_name}/4a-stats",
    )
)
# Standalone slurm runs (4a_submit_evaluation.sh) write to
# input/slurm-scripts/results/<var>/<run>/ instead, e.g.:
# stats_dir = Path("input/slurm-scripts/results/t2m/cerra-germany-0.05x0.05-sorted")

# --- raw files for the interactive point/area time series ------------------
var = "t2m"  # config name; the NetCDF variable name is derived via VAR_FILE_DEFAULTS
path_nn = f"{evaluation_dirs}/{config_name}/{run_name}/data"
path_original = "/work/uo1075/u244085/cerra_data/t2m/"
file_pattern_nn = "cerra_dm_{var}_ger_{year}_crai_output.nc"
file_pattern_ref = "cerra_dm_{var}_ger_{year}.nc"
year_min, year_max = 2016, 2025
point_lat, point_lon, point_name = 53.5511, 9.9937, "Hamburg"  # 4a default point

print("4a stats dir:", stats_dir.resolve())


In [ ]:
maps_files = sorted(stats_dir.glob("*_maps.nc"))
stats_files = sorted(stats_dir.glob("*_stats.txt"))
png_files = sorted(stats_dir.glob("*.png"))
print(f"{len(maps_files)} maps file(s), {len(stats_files)} stats file(s), "
      f"{len(png_files)} figure(s) in {stats_dir}")
for f in maps_files + stats_files:
    print("  ", f.name)
assert maps_files, (
    f"No *_maps.nc found in {stats_dir} - run the 4a analysis first "
    "(crai-evaluate --run-4a-stats or crai-evaluate-4a)"
)


In [ ]:
for f in stats_files:
    print(f"===== {f.name} =====")
    print(f.read_text())


In [ ]:
maps = xr.open_dataset(maps_files[-1])  # latest, if several runs are present
print("maps variables:", list(maps.data_vars))
maps


In [ ]:
key_maps = [
    m for m in ("bias", "rmse", "explained_variance", "correlation",
                "variance_ratio", "fraction_gap_closed")
    if m in maps.data_vars
]
cmaps = {"bias": "RdBu_r", "explained_variance": "RdBu_r", "correlation": "RdBu_r"}
ncols = 2
nrows = int(np.ceil(len(key_maps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows), squeeze=False)
for ax, name in zip(axes.ravel(), key_maps):
    maps[name].plot(ax=ax, cmap=cmaps.get(name, "viridis"))
    ax.set_title(name)
for ax in axes.ravel()[len(key_maps):]:
    ax.axis("off")
fig.suptitle(maps_files[-1].name)
plt.tight_layout()
plt.show()


In [ ]:
extra_maps = sorted(v for v in maps.data_vars if v.startswith("error_q"))
extra_maps += [v for v in maps.data_vars if v.endswith("_anomaly")]
if extra_maps:
    ncols = 3
    nrows = int(np.ceil(len(extra_maps) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows), squeeze=False)
    for ax, name in zip(axes.ravel(), extra_maps):
        maps[name].plot(ax=ax, cmap="RdBu_r")
        ax.set_title(name)
    for ax in axes.ravel()[len(extra_maps):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("no error-quantile / anomaly maps (was --deseasonalize used?)")


In [ ]:
from IPython.display import Image, display

groups = {
    "ROC / AUC": [p for p in png_files if "_roc" in p.name],
    "QQ plot": [p for p in png_files if "_qq" in p.name],
    "point/area time series": [p for p in png_files if "timeseries" in p.name],
}
shown = {q for files in groups.values() for q in files}
groups["map + histogram panels"] = [p for p in png_files if p not in shown]
for title, files in groups.items():
    if not files:
        continue
    print(f"--- {title} ---")
    for p in files:
        print(p.name)
        display(Image(filename=str(p)))


## Interactive point/area time series

Recomputed from the raw per-year files with the packaged `evaluate_4a` helpers (the same `load_pair` / `build_point_mask` the 4a script uses). Change `point_lat` / `point_lon` / `year_min` / `year_max` / `path_original` in the config cell above and re-run this cell. (The `evaluate_4a` import switches matplotlib to the headless Agg backend, so the cell re-enables `%matplotlib inline` right after the import.)


In [ ]:
from climatereconstructionai import evaluate_4a as e4a

# evaluate_4a forces the headless Agg backend on import; re-enable inline plotting:
%matplotlib inline

var_file = e4a.VAR_FILE_DEFAULTS.get(var, var)
chunks = {"time": 30}
nn_series, ref_series = [], []
for year in range(year_min, year_max + 1):
    file_nn = e4a.pattern_path(path_nn, file_pattern_nn, var, year)
    file_ref = e4a.pattern_path(path_original, file_pattern_ref, var, year)
    if not (Path(file_nn).is_file() and Path(file_ref).is_file()):
        print(f"skip {year}: missing {file_nn} or {file_ref}")
        continue
    nn, ref = e4a.load_pair(str(file_nn), str(file_ref), chunks, var_file)
    with xr.open_dataset(str(file_ref)) as ds0:
        p_lat, p_lon = e4a.detect_spatial_names(ds0)
        point_mask = e4a.build_point_mask(ds0, p_lat, p_lon, point_lat, point_lon, 0.0)
    nn_series.append(nn.where(point_mask).mean(dim=[p_lat, p_lon]).compute())
    ref_series.append(ref.where(point_mask).mean(dim=[p_lat, p_lon]).compute())

if not nn_series:
    print("No raw NN/reference files found - check path_nn / path_original / "
          "year range in the config cell.")
else:
    nn_ts = xr.concat(nn_series, dim="time")
    ref_ts = xr.concat(ref_series, dim="time")
    fig, (ax, ax_err) = plt.subplots(
        2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
    )
    ax.plot(ref_ts["time"], ref_ts, color="C1", lw=0.8, label="reference")
    ax.plot(nn_ts["time"], nn_ts, color="C0", lw=0.8, label="NN")
    ax.set_ylabel(var_file)
    ax.set_title(f"{point_name} ({point_lat}, {point_lon}) - nearest grid cell")
    ax.legend(frameon=False)
    ax_err.plot(nn_ts["time"], nn_ts - ref_ts, color="C3", lw=0.8)
    ax_err.axhline(0, color="k", lw=1)
    ax_err.set_ylabel("error (NN - reference)")
    ax_err.set_xlabel("time")
    plt.tight_layout()
    plt.show()
